In [1]:
import sys
import os
import torch
import spacy
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Add parent directory to path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

# Ensure spacy model is downloaded (run this in terminal if needed: python -m spacy download en_core_web_sm)
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Downloading Spacy model...")
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print(f"System ready. GPU Available: {torch.cuda.is_available()}")

System ready. GPU Available: True


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# Use float16 for speed if on GPU
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_name = "google/flan-t5-base"

print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name, 
    torch_dtype=torch_dtype
).to(device)

print("Model loaded.")

Loading google/flan-t5-base...
Model loaded.


In [5]:
def needs_decontextualization(text: str) -> bool:
    """
    Paper-Aligned Logic:
    1. Check for pronouns (He, She, It, They).
    2. Check for demonstratives (This, That, These).
    3. Check for starting conjunctions (But, And, However) - implies connection to prev sentence.
    """
    doc = nlp(text)
    
    # 1. Pronouns & Demonstratives
    target_words = {"this", "that", "these", "those", "he", "she", "it", "they", "his", "her", "its", "their"}
    for token in doc:
        if token.lower_ in target_words:
            return True
        if token.pos_ == "PRON":
            return True
            
    # 2. Starting Conjunctions (Context dependency)
    # If the sentence starts with 'But', 'However', 'Therefore', it relies on previous text.
    if doc[0].text.lower() in ["but", "however", "therefore", "and", "so"]:
        return True
            
    return False

# Test
tests = [
    "The economy is stable.", # False
    "However, it might crash soon.", # True (Conjunction + Pronoun)
    "This is unacceptable.", # True (Demonstrative)
    "Biden signed the bill." # False
]
for t in tests:
    print(f"'{t}' -> Needs Decontext? {needs_decontextualization(t)}")

'The economy is stable.' -> Needs Decontext? False
'However, it might crash soon.' -> Needs Decontext? True
'This is unacceptable.' -> Needs Decontext? True
'Biden signed the bill.' -> Needs Decontext? False


In [7]:
def resolve_coreference(sentence: str, full_context: str) -> str:
    """
    Rewrites the sentence to be standalone.
    """
    # 1. Filter
    if not needs_decontextualization(sentence):
        return sentence

    # 2. Prompt (Aligned with Paper's "Rewrite" objective)
    # We pass the full context but the model attends to what's needed.
    # In a real app, 'full_context' should be just the prev 1-3 sentences.
    prompt = (
        f"Rewrite this sentence to be fully explicit and standalone. "
        f"Resolve any pronouns or references using the context.\n\n"
        f"Context: {full_context}\n\n"
        f"Sentence: {sentence}\n\n"
        f"Output:"
    )
    
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids, 
            max_length=128, 
            num_beams=4, 
            early_stopping=True
        )
    
    resolved_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # 3. Sanity Check
    # If output is broken or empty, return original
    if not resolved_text or len(resolved_text) < 10:
        return sentence
        
    return resolved_text



In [ ]:
# Simulating a snippet where context is crucial
article_snippet = """
The new health bill was debated in the Senate yesterday.
Critics argued it would increase the deficit by billions.
However, supporters claimed it was necessary for reform.
"""

target_claims = [
    "Critics argued it would increase the deficit by billions.",
    "However, supporters claimed it was necessary for reform."
]

print(f"{'ORIGINAL':<60} | {'RESOLVED'}")
print("-" * 100)

for claim in target_claims:
    resolved = resolve_coreference(claim, article_snippet)
    print(f"{claim:<60} | \033[92m{resolved}\033[0m")